In [0]:
from pyspark.sql import SparkSession
#from pyspark.sql.functions import col, desc, row_number, rank, dense_rank, sum 
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
df = spark.read.csv(
    "s3://sinac-bronze/2008_2013/sinac2008DatosAbiertos.csv",
    header = True,
    inferSchema = True)

In [0]:
df.show(2)

In [0]:
df.printSchema()

In [0]:
# Amount of births per state with ranking
df_states = df.groupBy('edo_captura').count().orderBy('count', ascending = False)

# rank the states by the count
window_rank = Window.orderBy(F.desc('count'))

#Apply window function to the ranked states
ranked_df_states = df_states.withColumn("Ranking", F.dense_rank().over(window_rank))

ranked_df_states.show(32)


In [0]:
df.select('procedimiento_utilizado').distinct().show()

In [0]:
# Group the entries by how many procedures were made 
df_procedures = df.groupBy('edo_captura', 'procedimiento_utilizado').count().orderBy('edo_captura', "procedimiento_utilizado")

# Add a window function to partition the data 
window_spec = Window.partitionBy('edo_captura')

df_procedures = df_procedures.withColumnRenamed('count', 'procedimiento_conteo' )

# Add the column with the sum for each state
df_procedures = df_procedures.withColumn(
    'edo_conteo',
    F.sum('procedimiento_conteo').over(window_spec)
)

# Add the column with the percentage for each procedure
df_procedures = df_procedures.withColumn(
    'porcentaje_prevalencia',
    F.concat(F.format_number((F.col('procedimiento_conteo') / F.col('edo_conteo')) * 100, 2), F.lit('%'))
)

# Show the data 
df_procedures.show(n=1000, truncate=False)

In [0]:

# First, create your grouped dataframe
df_procedures = df.groupBy('edo_captura', 'procedimiento_utilizado').count()

# Add a window to calculate total per state
window_spec = Window.partitionBy('edo_captura')

# Add the total count per state
df_procedures = df_procedures.withColumn(
    'total_edo_count', 
    F_sum('count').over(window_spec)
).orderBy('edo_captura', 'procedimiento_utilizado').show()